# Building an Shopping Copilot with KumoRFM

In this example we will use KumoRFM to power an ecommerce platform shopping assistant.

In [ ]:
!pip install -qU \
    "datasets>=4.0.0" \
    "graphai-lib==0.0.10rc3" \
    "ipykernel>=6.30.1" \
    "ipywidgets>=8.1.7" \
    "kumoai==2.7.0" \
    "openai>=1.99.9"

## Building the Ecommerce Prediction Engine

We'll be using KumoRFM to power our shopping predictions. To make calls to KumoRFM an API key is required, you can a free API key via the widget below:

In [1]:
import os
from kumoai.experimental import rfm

if not os.environ.get("KUMO_API_KEY"):
    rfm.authenticate()

Opening browser page to automatically generate an API key...


[2025-09-11 18:21:02 - kumoai:298 - INFO] Generated token "sdk-jamess-macbook-pro.local-2025-09-11-18-20-58-Z" and saved to KUMO_API_KEY env variable


In [2]:
rfm.init(api_key=os.environ["KUMO_API_KEY"])

[2025-09-11 18:21:03 - kumoai:203 - INFO] Successfully initialized the Kumo SDK against deployment https://kumorfm.ai/api, with log level INFO.


We're going to use a sample of the H&M ecommerce dataset. The sample is available on Hugging Face Datasets at [jamescalam/hm-sample](https://huggingface.co/datasets/jamescalam/hm-sample). It includes three tables, the `customers` table with 1.1K rows, `articles` with 5K rows, and `transactions` with 15.7K rows.

In [3]:
from datasets import load_dataset

customers = load_dataset(
    "jamescalam/hm-sample", data_files="customers.jsonl", split="train"
)
customers

Dataset({
    features: ['customer_id', 'FN', 'Active', 'club_member_status', 'fashion_news_frequency', 'age', 'postal_code'],
    num_rows: 1100
})

In [4]:
articles = load_dataset(
    "jamescalam/hm-sample", data_files="articles.jsonl", split="train"
)
articles

Dataset({
    features: ['article_id', 'product_code', 'prod_name', 'product_type_no', 'product_type_name', 'product_group_name', 'graphical_appearance_no', 'graphical_appearance_name', 'colour_group_code', 'colour_group_name', 'perceived_colour_value_id', 'perceived_colour_value_name', 'perceived_colour_master_id', 'perceived_colour_master_name', 'department_no', 'department_name', 'index_code', 'index_name', 'index_group_no', 'index_group_name', 'section_no', 'section_name', 'garment_group_no', 'garment_group_name', 'detail_desc'],
    num_rows: 5000
})

In [5]:
transactions = load_dataset(
    "jamescalam/hm-sample", data_files="transactions.jsonl", split="train"
)
transactions

Dataset({
    features: ['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id'],
    num_rows: 15773
})

We'll read these into Kumo by first transforming our _data**sets**_ into Pandas _data**frames**_:

In [6]:
customers_df = customers.to_pandas()
articles_df = articles.to_pandas()
transactions_df = transactions.to_pandas()

Once we have our dataframes we will transform them into `rfm.LocalTable` objects. These are lightweight abstractions of pandas dataframes that allow us to interface our data with KumoRFM. We use the `.infer_metadata()` method to automatically infer what types of data we have in our tables:

In [7]:
customers = rfm.LocalTable(customers_df, name="customers").infer_metadata()
transactions = rfm.LocalTable(transactions_df, name="transactions").infer_metadata()
articles = rfm.LocalTable(articles_df, name="articles").infer_metadata()

Detected primary key 'customer_id' in table 'customers'
Detected time column 't_dat' in table 'transactions'
Detected primary key 'article_id' in table 'articles'


We can update the column types as needed like so:

In [8]:
# update semantic type of columns
customers["customer_id"].stype = "ID"
customers["age"].stype = "numerical"

# primary keys
customers.primary_key = "customer_id"
articles.primary_key = "article_id"

# time column
transactions.time_column = "t_dat"

Then we create the graph:

In [9]:
# select the tables
graph = rfm.LocalGraph(tables=[
    customers, transactions, articles
])
# link the tables
graph.link(src_table="transactions", fkey="customer_id", dst_table="customers")
graph.link(src_table="transactions", fkey="article_id", dst_table="articles")

LocalGraph(
  tables=[
    customers,
    transactions,
    articles,
  ],
  edges=[
    transactions.customer_id ⇔ customers.customer_id,
    transactions.article_id ⇔ articles.article_id,
  ],
)

Now we setup our KumoRFM model on our graph:

In [10]:
model = rfm.KumoRFM(graph=graph)

Output()

Now we can make predictions, let's see how likely one of our products are to be purchased over the next 30 days:

In [11]:
article_id = articles_df.iloc[0].article_id.item()
article_id

675662003

In [12]:
# forecast 30-day product demand for specific item/article
df = model.predict(
    f"PREDICT SUM(transactions.price, 0, 30, days) FOR articles.article_id={article_id}"
)
display(df)

Output()

,ENTITY,ANCHOR_TIMESTAMP,TARGET_PRED
0,675662003,1600732800000,0.000026


We have a pretty low likelihood of that product being purchased over the next 30 days. Let's see how likely two of our customers are to be purchases something over the next 90 days.

In [13]:
csample = customers_df.iloc[:2].customer_id.tolist()
csample

['1935b6baf9d28d1f19b7ffad18a9da418954a9bf38f59336f2f86d7a5615d1d2',
 '75ebdc56559b1f2739ce5832bd85a921ba827c72383135bdcc08a616d320e948']

In [14]:
# predict likelihood of two specific users not ordering in the next 90 days
df = model.predict(
    "PREDICT COUNT(transactions.*, 0, 90, days)=0 "
    f"FOR customers.customer_id IN ('{csample[0]}', '{csample[1]}')"
)
display(df)

Output()

,ENTITY,ANCHOR_TIMESTAMP,TARGET_PRED,False_PROB,True_PROB
0,1935b6baf9d28d1f19b7ffad18a9da418954a9bf38f593...,1600732800000,False,0.666437,0.333563
1,75ebdc56559b1f2739ce5832bd85a921ba827c72383135...,1600732800000,False,0.688486,0.311514


Now the next step is to build an AI agent that can do this for us.

## Building the Agent

In [15]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") or \
    getpass("Enter your OpenAI API key: ")

We then generate completions like so:

In [16]:
from openai import AsyncOpenAI

client = AsyncOpenAI()

response = await client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "user", "content": "Tell me something interesting about GNNs"}
    ],
    stream=True,
)

async for chunk in response:
    if (token := chunk.choices[0].delta.content) is not None:
        print(token, end="", flush=True)

Sure! One interesting aspect of Graph Neural Networks (GNNs) is their ability to **capture complex relational structures in data that traditional neural networks struggle with**. Unlike standard neural networks that operate on fixed-size vectors (like images or sequences), GNNs work directly on graphs, which are data structures composed of nodes (entities) connected by edges (relationships).

What makes GNNs particularly fascinating is their **message-passing mechanism**, where each node iteratively aggregates and transforms information from its local neighbors. This approach allows GNNs to learn rich, context-aware representations of nodes, edges, or entire graphs, making them powerful for tasks such as social network analysis, molecular chemistry (predicting properties of molecules), recommendation systems, and even reasoning over knowledge graphs.

Moreover, because graphs can represent non-Euclidean data—like social connections or road networks—GNNs open the door to applying deep l

Now we setup our agent with tools. We use the "no-framework" framework [GraphAI](https://docs.aurelio.ai/graphai/get-started/introduction).

Using this library we are expected to create our own tool functions, LLM API calls, etc. The library primarily acts as a graph execution framework _without_ any AI abstractions. With that in mind we will first define _two_ tools for our agent.

### Tool 1: Query Dataframes

The first tool runs a namespace `exec` instance allowing our LLM to run python code against our pandas dataframes.

With `graphai` we typically define tools with two components, a pydantic `BaseModel` to outline the tool schema for our agent, and the python function that will be executed when the tool is called.

In [17]:
import json
import pandas as pd
from pydantic import BaseModel, Field
from graphai import node
from graphai.callback import EventCallback


class QueryDataframes(BaseModel):
    """Execute simple filtered queries on the ecommerce dataframes. Will execute code in
    a namespace with the following dataframes:
    
    - transactions_df
    - articles_df
    - customers_df
    
    You can also access pandas library via `pd` for dataframe operations. Ensure you use
    assign the results you need to the `out` variable, otherwise nothing will be returned
    as this will be run with `exec()`. After execution we access the `out` variable and
    return it to you.

    If outputting a dataframe, you must use the .to_markdown() method to output an easily
    readable markdown table.
    """
    query: str = Field(..., description="The python code to execute")

@node(stream=True)
async def query_dataframes(input: dict, state: dict, callback: EventCallback) -> dict:
    try:
        tool_call_args = json.loads(state["events"][-1]["tool_calls"][0]["function"]["arguments"])
        # get dataframes, pandas, and set `out` to None
        namespace = {
            "transactions_df": state["transactions_df"],
            "articles_df": state["articles_df"],
            "customers_df": state["customers_df"],
            "pd": pd,
            "out": None,
        }
        # grab query from LLM to be executed
        query = tool_call_args.get("query")
        if not query:
            raise ValueError("No query provided")
        # remove escaped newlines as it frequently breaks the query
        query = query.replace("\\n", "\n")
        # execute query within predefined namespace
        exec(query, namespace)
        # pull out the `out` value
        out = namespace.get("out")
        if out is None:
            out = "No result returned via the `out` variable"
        content = [{"type": "text", "text": json.dumps(out, default=str)}]
    except Exception as e:
        content = [{
            "type": "text",
            "text": (
                f"Error executing query: {str(e)}. "
                "Please fix your query and trying again."
            )
        }]
    # stream tool output
    await callback.acall(
        type="tool_output",
        params={
            "id": state["events"][-1]["tool_calls"][0]["id"],
            "name": "predict_customer_purchase",
            "arguments": tool_call_args,
            "output": content[0]["text"]
        }
    )
    # Add tool call event to state
    event = {
        "role": "tool",
        "content": content,
        "tool_call_id": state["events"][-1]["tool_calls"][0]["id"]
    }
    state["events"].append(event)
    return {"input": {}}

[2025-09-11 18:21:38 - graphai.utils:160 - WARNING] Function query_dataframes has no docstring


### Tool 2: Query KumoRFM

The second tool will provide access to KumoRFM's PQL queries. For this tool to work, we need to add some guidelines on how to use it for our agent. We'll first grab those, the full prompt used can be [found here](https://github.com/jamescalam/ecommerce-agent/blob/main/api/pluto/prompts/developer.py).

In [18]:
import requests

pql_file = requests.get(
    "https://raw.githubusercontent.com/jamescalam/ecommerce-agent/refs/heads/main/api/pluto/prompts/developer.py"
).text
# strip first and last two lines as they contain python boilerplate
pql_reference = "\n".join(pql_file.split("\n")[1:-2])
print(pql_reference[:200])

# KumoRFM Predictive Query Language (PQL) Reference

## Overview

Predictive Query Language (PQL) is KumoRFM's declarative SQL-like syntax for defining predictive modeling tasks using the foundation m


Given the size of these guidelines we'll insert them directly into our system/developer message rather than our tool description.

Now we define the KumoRFM tool like so:

In [19]:
class KumoRFM(BaseModel):
    """This tool allows you to write any PQL query to the KumoRFM model.
    """
    query: str = Field(..., description="The PQL query to predict")

@node(stream=True)
async def kumorfm(input: dict, state: dict, callback: EventCallback) -> dict:
    try:
        tool_call_args = json.loads(state["events"][-1]["tool_calls"][0]["function"]["arguments"])
        query = tool_call_args.get("query")
        if not query:
            raise ValueError("No query provided")
        
        df = state["kumorfm"].predict(query)
        out = df.to_dict(orient="records")
        content = [{"type": "text", "text": json.dumps(out)}]
    except Exception as e:
        content = [{"type": "text", "text": str(e)}]
    # stream tool output
    await callback.acall(
        type="tool_output",
        params={
            "id": state["events"][-1]["tool_calls"][0]["id"],
            "name": "predict_customer_purchase",
            "arguments": tool_call_args,
            "output": content[0]["text"]
        }
    )
    event = {
        "role": "tool",
        "content": content,
        "tool_call_id": state["events"][-1]["tool_calls"][0]["id"]
    }
    state["events"].append(event)
    return {"input": {}}

[2025-09-11 18:21:38 - graphai.utils:160 - WARNING] Function kumorfm has no docstring


For our LLM to be able to read our tool schemas, we will be using the built-in `FunctionSchema` method. We can use this to consume our pydantic base models and later output them into an OpenAI-friendly schema format.

In [20]:
from graphai.utils import FunctionSchema

query_df_schema = FunctionSchema.from_pydantic(QueryDataframes)
query_df_schema.name = "query_dataframes"
kumorfm_schema = FunctionSchema.from_pydantic(KumoRFM)
kumorfm_schema.name = "kumorfm"

tools = [query_df_schema, kumorfm_schema]

The schemas can then be created using the `to_openai` method (when using OpenAI models).

In [21]:
tools[1].to_openai(api="completions")

{'type': 'function',
 'function': {'name': 'kumorfm',
  'description': 'This tool allows you to write any PQL query to the KumoRFM model.\n    ',
  'parameters': {'type': 'object',
   'properties': {'query': {'description': 'The PQL query to predict',
     'type': 'string'}},
   'required': ['query']}}}

### Building the Graph

Graphs are constructed from nodes and edges, with various special nodes and edges within that broader structure. For our use-case we don't need to dive into anything too exotic. All we need to do is define our nodes, and construct our graph to join those together.

#### Our Nodes

The graph will consist of _five_ total nodes, two of those we have already defined with our tools. The remaining three are:

- `llm` router node will contain the logic for calling our LLM and handling our LLM's tool-calling decisions.
- `start` and `end` nodes are `graphai`-specific boilerplate, they act as the entry and exit points of our graph

We will first define the `llm` router:

In [22]:
from graphai import router

@router(stream=True)
async def llm(input: dict, state: dict, callback: EventCallback) -> dict:
    # get client initialized in lifespan
    client = state["client"]
    # call openai (or another provider as preferred)
    stream = await client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=state["events"],
        tools=[x.to_openai(api="completions") for x in tools],
        stream=True,
        seed=9000,  # keep consistent results
        parallel_tool_calls=False,
    )
    direct_answer: str = ""
    tool_call: dict = {}
    tool_call_args = ""
    async for chunk in stream:
        if (token := chunk.choices[0].delta.content) is not None:
            # this handles direct text output
            direct_answer += token
            await callback.acall(token=token)
        # handle tool calls
        tool_calls_out = chunk.choices[0].delta.tool_calls
        if tool_calls_out and (tool_name := tool_calls_out[0].function.name) is not None:
            # this handles the initial tokens of a tool call
            tool_call["id"] = tool_calls_out[0].id
            tool_call["name"] = tool_name
            # we can return the tool name
            await callback.acall(
                type="tool_call",
                params=tool_call
            )
        elif tool_calls_out and (tool_args := tool_calls_out[0].function.arguments) is not None:
            # this handles the arguments of a tool call
            tool_call_args += tool_args
            # we can output these too
            await callback.acall(
                type="tool_args",
                params={
                    **tool_call,
                    "arguments": tool_args
                }
            )
    if direct_answer:
        # if we got a direct answer we create a standard assistant message
        state["events"].append(
            {
                "role": "assistant",
                "content": direct_answer,
            }
        )
        # choice controls the next node destination
        choice = "end"
    elif tool_call:
        # if we got a tool call we create an assistant tool call message
        state["events"].append(
            {
                "role": "assistant",
                "tool_calls": [{
                    "id": tool_call["id"],
                    "type": "function",
                    "function": {
                        "name": tool_call["name"],
                        "arguments": tool_call_args,
                    }
                }]
            }
        )
        choice = tool_call["name"]
    return {"input": input, "choice": choice}

[2025-09-11 18:21:46 - graphai.utils:160 - WARNING] Function llm has no docstring


And now our two boilerplate `start` and `end` nodes:

In [23]:
@node(start=True)
async def start(input: dict) -> dict:
    return {"input": input}

@node(end=True)
async def end(input: dict, state: dict) -> dict:
    return {"output": state["events"]}

[2025-09-11 18:21:47 - graphai.utils:160 - WARNING] Function start has no docstring
[2025-09-11 18:21:47 - graphai.utils:160 - WARNING] Function end has no docstring


#### Constructing the Graph

Our broader graph contains the logic that connects our various nodes and defines the initial state of the workflow. We will first define our state, which will consist of our initial system/developer message, our KumoRFM instance, and the three H&M dataframes.

We will begin by defining the developer message:

In [24]:
dev_message = {
    "role": "developer",
    "content": (
        "You are a helpful assistant that uses the various tools and "
        "KumoRFM integration to answer the user's analytics questions "
        "about our H&M ecommerce dataset."
        "\n"
        "When answering questions, you may use the various tools "
        "multiple times before answering to the user. You should aim "
        "aim to have all of the information you need from the tools "
        "before answering the user."
        "\n"
        "There is a limit of 30 steps to each interaction, measured "
        "as the number of tool calls made between the user's most "
        "recent message and your response to the user. Keep that limit "
        "in mind but ensure you are still thorough in your analysis."
    )
}

And now our initial state:

In [25]:
initial_state = {
    "events": [dev_message],
    "kumorfm": model,
    "transactions_df": transactions_df,
    "articles_df": articles_df,
    "customers_df": customers_df,
    "client": client
}

This state can be added to our graph using the `set_state` method. Alongside this we will also be adding the various nodes and routers to our graph with `add_node` and `add_router`. We then set all edges with `add_edge`. Finally, once our graph is fully defined we `compile` it.

In [26]:
from graphai import Graph

# create graph
graph = (
    Graph(max_steps=30)
    .set_state(initial_state)
    .add_node(start)
    .add_node(llm)
    .add_node(kumorfm)
    .add_node(query_dataframes)
    .add_node(end)
    .add_router(
        sources=[start],
        router=llm,
        destinations=[
            kumorfm,
            query_dataframes,
            end
        ]
    )
    .add_edge(kumorfm, llm)
    .add_edge(query_dataframes, llm)
    .add_edge(llm, end)
    .compile()
)

## Using our Agent

The agent is now fully defined and we can start using it. We call it with `await graph.execute` like so:

In [27]:
cb = EventCallback()
# add our input message to the state
graph.update_state({
    "events": [
        *graph.state["events"],
        {
            "role": "user",
            "content": f"Can you predict the demand for article {article_id} over the next 30 days"
        }
    ]
})
# now execute
out = await graph.execute({"input": {}}, callback=cb)

# and (optionally) stream the output
async for event in cb.aiter():
    if str(event.type) == "callback":
        # this indicates direct text output
        print(event.token, end="")
    elif event.type == "tool_call":
        # this indicates the first event in a tool call
        # this contains tool name and ID
        print(event.params["name"])
    elif event.type == "tool_args":
        # this indicates the arguments of a tool call
        print(event.params["arguments"], end="")
    elif event.type == "tool_output":
        # this indicates the output of a tool call
        # these can be very long so we'll avoid printing them
        # but feel free to try
        #print(event.params["output"])
        pass

kumorfm
{"query":"PREDICT sales FROM Article WHERE article_id=675662003 FOR NEXT 30 DAYS"}kumorfm
{"query":"PREDICT demand FROM article WHERE article.article_id=675662003 FOR EACH DATE IN NEXT 30 DAYS"}kumorfm
{"query":"PREDICT quantity FROM transaction WHERE article_id=675662003 FOR EACH day IN NEXT 30 DAYS"}kumorfm
{"query":"PREDICT count(*) FROM transaction WHERE article_id=675662003 FOR EACH day IN NEXT 30 DAYS"}kumorfm
{"query":"PREDICT COUNT() ON transaction WHERE transaction.article_id = 675662003 FOR EACH transaction.date BETWEEN TODAY() AND ADD_DAYS(TODAY(), 30)"}kumorfm
{"query":"PREDICT COUNT(*) FROM transaction WHERE article_id = 675662003 FOR EACH date IN DATE_RANGE(TODAY(), ADD_DAYS(TODAY(), 30))"}kumorfm
{"query":"PREDICT count FROM transaction.article_sales WHERE article_id=675662003 FOR 30 DAYS"}kumorfm
{"query":"PREDICT count FROM transaction WHERE article.article_id=675662003 FOR EACH SEQUENCE day IN NEXT 30 DAYS"}kumorfm
{"query":"PREDICT number_of_sales ON transact

Let's write a helper function for our chat:

In [ ]:
async def chat(content: str):
    cb = EventCallback()
    graph.update_state({
        "events": [
            *graph.state["events"],
            {"role": "user", "content": content}
        ]
    })

    _ = await graph.execute({"input": {}}, callback=cb)
    
    async for event in cb.aiter():
        if str(event.type) == "callback":
            # this handles direct text output
            print(event.token, end="")
        elif event.type == "tool_call":
            # this indicates the first event in a tool call
            # this contains tool name and ID
            print(event.params["name"])
        elif event.type == "tool_args":
            # this indicates the arguments of a tool call
            print(event.params["arguments"], end="")
        elif event.type == "tool_output":
            # this indicates the output of a tool call
            # these can be very long so we'll avoid printing them
            # but feel free to try
            #print(event.params["output"])
            pass
        

In [ ]:
await chat(
    "What other useful info can you give me? I'm preparing our monthly marketing "
    "emails"
)

In [ ]:
await chat("Can you help me find customers likely to churn?")

In [ ]:
await chat("Can you get a sample of 50 customers?")

In [ ]:
await chat("Okay, but are those recently active customers?")

In [ ]:
await chat(
    "Okay let's use these, let's filter down to the most likely to churn who also have past "
    "purchase history with us"
)

In [ ]:
await chat("can you give me the top 3?")

In [ ]:
await chat(
    "let's write a personalized email to the first customer on that list - using what "
    "we know about their past purchases and predicted most likely future purchases"
)

In [ ]:
await chat("that feels a little too obviously automated, can you make it more natural?")

In [ ]:
await chat(
    "that's better, but let's be specific about what we think they'd like? "
    "We should include the product names and add the article IDs in square brackets and "
    "an image of the product will appear in the email"
)

In [ ]:
await chat("no I mean let's add in the article IDs for the new products we predicted they'd like")

In [ ]:
await chat("that's perfect thanks! Can you do the same for customer 3?")

In [ ]:
await chat("yes let's write the email")

---